# Ejercicio AirBnB: Extracción de entidades

En este cuaderno vamos a trabajar con un dataset de AirBnB de la ciudad de Oporto. Se puede encontrar más información sobre el dataset y otras implementaciones en [Porto](https://github.com/Vasallo94/Porto). 

El dataset contiene información sobre las características de las viviendas, su localización, el precio, el número de comentarios, etc.

El objetivo de este ejercicio es poder analizar los comentarios de los usuarios mediante LLMs y poder extraer información relevante de los mismos para su análisis posterior.

## Primera parte

Vamos a utilizar el archivo listings1_cleaned.csv que contiene información sobre las viviendas de AirBnB en Oporto.

1. Filtra el dataset con una función que seleccione las 10/20 reseñas con mayor longitud (de string) en la columna descripción (las que consideraríamos las más 'relevantes')
2. Elabora un prompt de tal forma que un LLM sea capaz de extraer un json con las siguientes entidades de las entradas de la tabla filtrada:
```json
{
    "name": str,
    "location": str,
    "main_characteristics": str,
    "type": str,
    "size": str,
    "capacity": str,
    "key_amenities": list,
    "proximity_highlights": list,
}
```
3. Puedes elegir el proveedor que quieras, Gemini de Google o cualquiera de los modelos de Groq

In [1]:
import pandas as pd
from google.genai import types as genai_config_types
import os
import getpass
from google import genai

# Configurar la API Key de Gemini o configura el cliente de groq! Usa el proveedor que quieras
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter google AI api key : ")

if not os.environ.get("GOOGLE_MODEL"):
    os.environ["GOOGLE_MODEL"] = getpass.getpass("Enter google AI model name : ")

if not os.environ.get("BATCH_SIZE"):
    os.environ["BATCH_SIZE"] = '20'

GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
GOOGLE_MODEL = os.environ['GOOGLE_MODEL']
BATCH_SIZE = int(os.environ["BATCH_SIZE"])

google_client = genai.Client(api_key=GOOGLE_API_KEY)


#### Dataset modificado
En este caso, se ha realizado un proceso de filtrado del dataset para que la información que reciba el modelo sea más precisa. En nuestro caso, realizamos las siguientes operaciones:
- Filtramos el dataset para obtener las columnas que nos aportan la informaciópn necesaria en este ejercicio, que son: 'name', 'description', 'property_type', 'room_type', 'amenities', 'latitude', 'longitude', 'geographical_group', 'accommodates', 'bathrooms', 'bedrooms', 'beds'

- Después, aplicamos una función auxiliar para obtener un numero, a elección (por defecto, 20), de filas las cuales son las filas con la descripción más larga. En este caso, no evaluamos aquellas descripciones vacias o nulas

In [2]:
original_data_p1 = pd.read_csv(
    filepath_or_buffer="../../docs/listings1_cleaned.csv")
print("Data collected successfully")

filtered_columns=['name', 'description', 'property_type', 'room_type', 'amenities', 'latitude', 'longitude', 'neighbourhood', 'accommodates', 'bathrooms', 'bedrooms', 'beds']
filtered_data_p1 = original_data_p1[filtered_columns] 

Data collected successfully


In [3]:
def clean_special_characters(dataframe_column: pd.Series) -> pd.Series:
    """Remove all special characters from a string Series, keeping letters, digits, whitespace, and [. , - _]."""
    return dataframe_column.astype(str).str.replace(r"[^\w\s.,\-]", " ", regex=True)

def get_largest_reviews_filtered(dataframe: pd.DataFrame, comment_field: str = 'description', remove_special_characters:bool = True, size: int=20) -> pd.DataFrame:
    """Return first dataframe n rows with longest description values."""
    filtered_desc_col = dataframe[comment_field].dropna()
    filtered_desc_col = filtered_desc_col[filtered_desc_col.str.strip() != ""]
    
    if filtered_desc_col.empty:
        raise ValueError("Series has no valid (non-NaN, non-empty) string values")

    if remove_special_characters:
        dataframe[comment_field] = clean_special_characters( dataframe['description'])

    return dataframe.loc[filtered_desc_col.str.len().nlargest(size).index]


In [4]:
filtered_data_p1 = get_largest_reviews_filtered(dataframe=filtered_data_p1, size=BATCH_SIZE)

print(f"Length: {[len(value) for id, value in filtered_data_p1['description'].items()]}")
print(f"{len(filtered_data_p1)}")

Length: [1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]
10


In [7]:
response_schema_p1 = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "description": "Name of hotel"
            },
            "location": {
                "type": "string",
                "description": "Brief description of the hotel's location within the country"
            },
            "main_characteristics": {
                "type": "string",
                "description": "Brief description of what's included in a room ",
            },
            "type": {
                "type": "string",
                "description": "Field indicating the type of hotel in the review"
            },
            "size": {
                "type": "string",
                "description": "Brief description of how many people can sleep in a room at a time"
            },
            "capacity": {
                "type": "string",
                "description": "brief description of how many people can renta a room in total"
            },
            "key_aminities": {
                "type": "string",
                "description": "brief description of which aminities are included into the hotel"
            },
            "proximity_highlights": {
                "type": "string",
                "description": "A brief description of the points of interest in the area surrounding the hotel based on its location"
            }
        },
        "required": ["name", "location", "main_characteristics", "type", "size", "capacity", "key_aminities", "proximity_highlights"],
    }
}

instructions_content_p1 = ""
with open("../../docs/instructions/P1_1/system_instructions.txt", "r", encoding="utf-8") as f:
    instructions_content_p1 = f.read()

print("System instruccions read successfully")

datos_json_p1 = filtered_data_p1.to_json(orient="records", indent=2, force_ascii=False)

configuration_p1 = genai_config_types.GenerateContentConfig(
        system_instruction=instructions_content_p1,
        response_modalities=["TEXT"],
        temperature=1.0,
        top_k=250,
        top_p=0.8,
        seed=12345,
        response_mime_type="application/json",
        response_schema=response_schema_p1
)

System instruccions read successfully


In [8]:

promtp_p1 = f"""
    Extract information about airbnb hotels reviews and fill response with the response schema indicated
    Here you can find your information source about airbnb hotel reviews filtered: 
    
    {datos_json_p1}
"""

print(f"Prompt: {promtp_p1}")

response_p1 = google_client.models.generate_content(
    model=GOOGLE_MODEL,
    contents=promtp_p1,
    config=configuration_p1
)

Prompt: 
    Extract information about airbnb hotels reviews and fill response with the response schema indicated
    Here you can find your information source about airbnb hotel reviews filtered: 

    [
  {
    "name":"Magnificent Apartment in Miles End",
    "description":"Enter a private foyer and step into a superb designer space where the colors are relaxing shades of beige with taupe accents. Light floods through a wall of patio doors which open onto a balcony overlooking Saint Laurent Boulevard. br    br   CITQ  271283 br    br   This is a treasure behind a simple door. You enter, walk up a flight of stairs, enter a private foyer, and from there walk up into a superb designer space. br    br   The living room features elegant designer furnishings, art work. br    br   The colors are relaxing shades of beige with taupe accents. A wall of patio doors open onto a balcony overlooking St Laurent, affectionately known as The Main. The room fis bright, facing west. Curtains provide sh

In [9]:
print(f"Gemini response: {response_p1.text}")

Gemini response: [
  {
    "name": "Magnificent Apartment in Miles End",
    "location": "Le Plateau-Mont-Royal, Montreal",
    "main_characteristics": "Apartment for 2 people with 1 bedroom, 1 bed, and 1 bathroom",
    "type": "Entire rental unit",
    "size": "Space for 1 bed, suitable for compact stays",
    "capacity": "Up to 2 guests",
    "key_aminities": "Air conditioning, Wifi, Heating, Kitchen, Washer, Dryer, TV, Balcony",
    "proximity_highlights": "Located in the vibrant Le Plateau-Mont-Royal neighborhood, close to Saint Laurent Boulevard and local shops."
  },
  {
    "name": "Bohemian Loft Retreat in Montreal’s Old Port",
    "location": "Old Montreal, Ville-Marie",
    "main_characteristics": "Loft for 6 people with 1 bedroom, 2 beds, and 1.5 bathrooms",
    "type": "Entire loft",
    "size": "Space for 2 beds, suitable for compact stays",
    "capacity": "Up to 6 guests",
    "key_aminities": "Wifi, Air conditioning, Heating, Kitchen, Jacuzzi tub, Fireplace, Washer, Dry

## Parte 2

Realiza un análisis de los comentarios de los apartamentos con un LLM y extrae información relevante de los mismos.

Por ejemplo, puedes analizar los comentarios y extraer información sobre la limpieza, la ubicación, la relación calidad-precio, el sentimiento del comentario, etc. 

Igual que la parte 1 pero con un segundo dataset y con formato de salida a tu gusto

In [10]:
comentarios = pd.read_csv("../../docs/Airbnb_reviews_5000.csv")

comentarios_filtered_p2 = comentarios.copy()
print(f"{comentarios.head(-5)}")
print(f"{comentarios.columns}")

                                                   name      host_id  \
0                      LUXURY apartment t3 oporto antas   37249350.0   
1        Aida's Haven | Room&PrivateBath | St. Catarina   38365612.0   
2     Maritime Inspiration - One Bedroom Beach Apart...  147469727.0   
3              Central  charming Top floor - nice views   26222276.0   
4            Ribeira Oporto Apartment II (Renewed 2021)   35057317.0   
...                                                 ...          ...   
4990               Apartment in the centre with parking   74215308.0   
4991  Estudio7 D.Maria -7 Bridges - Porto-houses&suites   17289096.0   
4992  Casas Brancas w/ Balcony & free private parkin...  280564100.0   
4993                   Luiz Primeiro 00 | Premium Lofts   32914732.0   
4994  SãoMiguelApartments HistoricalCenter SuperiorA...    3469454.0   

       host_name        date  reviewer_id reviewer_name  \
0        Joaquim  2018-01-30  156292241.0        Silvia   
1      Alexandra 

In [69]:
len(comentarios_filtered_p2)

5000

In [11]:
print(f"Null comments: {comentarios_filtered_p2['comments'].isna().sum()}")
print(f"Empty comments: {(comentarios_filtered_p2['comments'] == '').sum()}")
# filtered_desc_col.str.len().nlargest(size).index
min_comment_length = comentarios_filtered_p2['comments'].str.len().nlargest(len(comentarios_filtered_p2)).min()
max_comment_length = comentarios_filtered_p2['comments'].str.len().nlargest(len(comentarios_filtered_p2)).max()
print(f"Min comment length: {min_comment_length}. Max comment length: {max_comment_length}")
print(f"Empty registers: {comentarios_filtered_p2.isna().all(axis=1).sum()}")

Null comments: 36
Empty comments: 0
Min comment length: 1.0. Max comment length: 2976.0
Empty registers: 18


<!-- Prompt injection o contenido que no aporte información según las columnas del archivo de entrada. En caso de no tener información o no se puede identificar -> Unkown 

Tipos de comentarios (multiples idiomas):
- Agradecimiento con una palabra: Analizar si esa palabra es negativa, neutra o positiva. Ejemplos:
    * bad, not, sucks, dont -> negative
    * OK, nice -> neutro
    * nice!, ok!, fantastic! -> positive
- Cualquier comentario que solo incluya simbolos o signos que no sean alfanumericos: No aplica
- Con etiquetas de html (sin tratamiento y obvia, posible prompt injection)
- Comentarios mal escritos (Entender que puede ser natural e intentar analizar el sentido del comentario si es posible.)
- Sentimiento del comentario positivo, negativio o neutro, 
- Agradecimientos a una persona en concreto. Analizar si el comentario contiene un nombre de una persona o se dirige al personal del hotel
- en todo momento, analiza el sentimiento del comentario para discernir entre: Positivo, neutro o negativo.
    Podria ser mezcla entre positivo con puntos negativos, tradúcelo como neutro

Language: 
- Idioma del registro. Puede no indicar un idioma concreto (OK, KO, ., NO). 
- Conocer idioma para saber analizar el registro.Si no indica idioma claro,
    deducirlo del comentario (campo "comments")

'reviewer_name', 'host_name': 
- En varios idiomas: No traducir 
- Con caracteres que no sea alfanumerico -> Unkown
- No quiero analizar si es un nombre o no, no tiene sentido ya que puede ser un nickname
- Puede ser dos nombres a la vez. Quedarte con los dos

'name':
- Caracteres extraños o imagenes: No contemplar ni tratar, solamente ignorarlo.
- Tener sentido comun e identificar si es un nombre real, puede ser ficticio o no es un nombre
- Contemplar que sea una descripción. Deducir de la descripción el hotel. -->

In [71]:
comentarios_filtered_p2.columns

Index(['name', 'host_id', 'host_name', 'date', 'reviewer_id', 'reviewer_name',
       'comments', 'language'],
      dtype='str')

In [12]:
# Remove empty registers
comentarios_filtered_p2.dropna(how = 'all', inplace=True)
print(f"Empty registers: {(comentarios_filtered_p2 == '').all(axis=1).sum()}")
len(comentarios_filtered_p2)

# Remove unsued columns
unused_columns = ['host_id', 'reviewer_id', 'date']
comentarios_filtered_p2.drop(columns=unused_columns, inplace=True)

Empty registers: 0


In [13]:
comentarios_filtered_p2 = get_largest_reviews_filtered(
    dataframe=comentarios_filtered_p2, 
    comment_field='comments', 
    remove_special_characters=False,
    size=BATCH_SIZE)
print(f"Length: {[len(value) for id, value in comentarios_filtered_p2['comments'].items()]}")

Length: [2976, 2392, 2226, 2011, 1928, 1846, 1817, 1808, 1795, 1781]


In [14]:
response_schema_p2 = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "hotel_name": {
                "type": "string",
                "description": "Name of hotel "
            },
            "host_name": {
                "type": "string",
                "description": "Name of people who were hopedate in the hotel"
            },
            "review_owner": {
                "type": "string",
                "description": "Person or people who made the hotel review",
            },
            "review_data": {
                "type": "object",
                "description": "Summarized comment about the review",
                "properties": {
                    "mentioned_person": {
                        "type": "string",
                        "description": "Name of a specific person referenced in the review, if any. Null if no individual is mentioned."
                    },
                    "highlights": {
                        "type": "string",
                        "description": "What stands out positively or negatively in the review about hotel, people, facilities, aminities or anything related to hotel"
                    },
                    "areas_to_improve": {
                        "type": "string",
                        "description": "What the review suggests should be improved, if exist"
                    }
                },
                "required": ["highlights"]
            },
            "is_client_satisfied": {
                "type": "boolean",
                "description": "Value that indicates whether the client expresses overall satisfaction in the review, based on positive sentiment or favorable remarks."
            }
        },
        "required": ["hotel_name", "host_name", "review_owner", "review_data", "is_client_satisfied"]
    }
}

instructions_content_p2 = ""
with open("../../docs/instructions/P1_2/system_instructions.txt", "r", encoding="utf-8") as f:
    instructions_content_p2 = f.read()

prompt_comfiguration_p2 = genai_config_types.GenerateContentConfig(
            system_instruction=instructions_content_p2,
            response_modalities=["TEXT"],
            temperature=1.0,
            top_k=250,
            top_p=0.8,
            seed=12345,
            response_mime_type="application/json",
            response_schema=response_schema_p2
        )

comentarios_data_p2 = comentarios_filtered_p2.to_json(orient="records", indent=2, force_ascii=False)
print(f"{len(comentarios_filtered_p2)}")
print(f"{comentarios_data_p2}")

10
[
  {
    "name":"FLH Porto Homey Flat with Balcony",
    "host_name":"Feels Like Home",
    "reviewer_name":"Ricardo",
    "comments":"This WOULD have been a 5 star review BUT:<br\/><br\/>I don't typically leave negative or critical reviews but given the potential for this space to be incredible, I wanted to provide some feedback... the space is actually beautiful and well-equipped (with a working washer\/dryer).  It would be very easy to make this space 5 stars and an ideal stay in Porto except for the following issues that the MANAGEMENT company is responsible for (they should just charge a couple extra euros and fix\/provide the following):<br\/><br\/>1. General maintenance\/awareness of the space (the toilet seat cover was completely falling off, the soap dispenser for the sink was broken, a shower towel was torn, a chair in the dining area was broken and unusable).<br\/>2. No shower soap or shampoo\/conditioner is provided (if this wasn't such a nice place, maybe I wouldn't ex

In [15]:
# prompt = """Extract information about airbnb hotels reviews and fill response with the response schema indicated. 
# Here you can find your information source about airbnb hotel reviews and must follow the instructions exactly as written.
# Keeping batch analysis input order, here you have the {number} / {batch_size} batch of information:

# {information_batch}
# """

prompt_p2 = f"""Extract information about airbnb hotels reviews and fill response with the response schema indicated. 
Here you can find your information source about airbnb hotel reviews filtered:

{comentarios_data_p2}
"""

# results = []
# for i in range(0, len(comentarios_filtered), BATCH_SIZE):
#     batch_comment = comentarios_filtered.iloc[i:i + BATCH_SIZE]
    
#     filled_prompt = prompt.format(
#         number=i, 
#         batch_size=BATCH_SIZE, 
#         information_batch="\n\n".join(f"Review {i+1}: {batch_comment}")
#     )

response_v2 = google_client.models.generate_content(
    model=GOOGLE_MODEL,
    contents=prompt_p2,
    config=prompt_comfiguration_p2
)

    # if response_v2.text is not None:
    #     print(f"WARN: Response in batch number {i} got null value ...")
    #     results.extend(json.loads(response_v2.text))


In [16]:
print(f"Gemini response: {response_v2.text}")

Gemini response: [
  {
    "hotel_name": "FLH Porto Homey Flat with Balcony",
    "host_name": "Feels Like Home",
    "review_owner": "Ricardo",
    "review_data": {
      "highlights": "The space is beautiful and well-equipped, with a working washer/dryer. However, there are issues with general maintenance such as a broken toilet seat, soap dispenser, torn towel, and a broken chair. Additionally, no shower soap or shampoo is provided, and laundry soap is not supplied for the washer/dryer. The check-in process requires picking up keys at a different location before going to the apartment, which is inconvenient. The management company's communication and problem-solving approach are also criticized for being lazy and inconvenient for guests."
    },
    "is_client_satisfied": false
  },
  {
    "hotel_name": "Oporto FR Cativo Flat",
    "host_name": "Fábio",
    "review_owner": "Nicola",
    "review_data": {
      "mentioned_person": "Fábio",
      "highlights": "The apartment building 